In [ ]:
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from common.preprocess import *

In [ ]:
""" cache directory. """
NOTEBOOK_CACHE = Path("__tmp") / "american_gut_USCA"
NOTEBOOK_CACHE.mkdir(exist_ok=True, parents=True)

""" Huggingface token """
with open("hf_token.txt", "rt") as f:
    token = f.readline().strip()
    assert len(token) > 0, "File 'hf_token.txt' should contain the hugging face token in the first line."
    os.environ["HF_TOKEN"] = token
    print("Successfully set huggingface token from file 'hf_token.txt'!")

# Dataset Files

File locations and metadata.

In [ ]:
data_base_dir = Path("/data/cctm/youn/human_microbiome_compendium")

project_metadata_file = data_base_dir / "projects.csv"
asv_sequence_file = data_base_dir / "obs_md.txt.zst"  # tsv format: ASV_NAME    ASV_SEQ, has a header.
abundance_table_dir = data_base_dir / "asv"
sample_metadata_file = data_base_dir / "sample_metadata.tsv"

assert asv_sequence_file.exists(), f"Expected {asv_sequence_file} to exist."
assert asv_sequence_file.is_file(), f"Expected {asv_sequence_file} to be a file."
assert abundance_table_dir.exists(), f"Expected {asv_sequence_file} to exist."
assert abundance_table_dir.is_dir(), f"Expected {abundance_table_dir} to be a directory."
assert sample_metadata_file.exists(), f"Expected {sample_metadata_file} to exist."
assert sample_metadata_file.is_file(), f"Expected {sample_metadata_file} to be a file."

""" Load the project metadata as a pandas dataframe. """
project_metadata = pd.read_csv(project_metadata_file, sep=',')
print("# projects:", project_metadata.shape[0])

""" Load the sample metadata as a pandas dataframe. """
sample_metadata = pd.read_csv(sample_metadata_file, sep='\t')
print("# samples:", sample_metadata.shape[0])

# Projects -- Target Subset by ID

In [ ]:
PROJECT_IDS = ["PRJEB11419"]


for project_id in PROJECT_IDS:
    print_project_info(project_id, sample_metadata)

# Sample & project filtering.

In [ ]:
project_subset, sample_subset, asv_seqs_subset, sample_max_num_asvs = filter_samples_and_asvs(
    project_metadata, sample_metadata,
    target_project_ids=set(PROJECT_IDS),
    abundance_table_dir=abundance_table_dir,
    asv_sequence_file=asv_sequence_file,
)

# Process 16S sequences.

Keep only sequences that are actually 16S. (some are 18s by accident!)

In [ ]:
ASV_SEQ_PROCESSING_DIR = NOTEBOOK_CACHE / "asv_16s_processing"
ASV_SEQ_PROCESSING_DIR.mkdir(exist_ok=True, parents=True)

"""
Note: defer_to_hpc option makes this pipeline print HPC instructions and raise an error.
Follow the directions, and re-run this cell.
"""
asv_seqs_subset = pipeline_16s_validation(
    asv_seqs_subset,
    cache_dir=ASV_SEQ_PROCESSING_DIR,
    blast_db=Path("/data/cctm/blast_dbs/core_nt/core_nt"),
    defer_to_hpc=True
)

In [ ]:
""" Run the alignment. """
asv_sequence_file_postblast = ASV_SEQ_PROCESSING_DIR / "asv_sequences.post_filter.fasta"
asv_align_file = ASV_SEQ_PROCESSING_DIR / "asv_alignment.fasta"

dict_to_fasta(asv_seqs_subset, asv_sequence_file_postblast)
run_mafft(asv_sequence_file_postblast, asv_align_file)

In [ ]:
""" Compute the max sequence length. """
MAX_ASV_SEQUENCE_LEN = max(len(s) for s in asv_seqs_subset.values())
print("Max ASV sequence length:", MAX_ASV_SEQUENCE_LEN)

""" Plot ASV length frequency histogram """
fig, ax = plt.subplots(1, 1)
ax.hist([len(s) for s in asv_seqs_subset.values()], bins=100)
ax.set_xlabel("ASV Sequence length")
ax.set_ylabel("Count")